# What Keeps a Reddit Rivalry Alive?

We study how negative references between Reddit communities change over time.
Our aim is to distinguish declining negativity while interaction continues from a lack of further observed contact.

## Research questions

1. After repeated negative references, does negativity persist, decline with continued contact, or give way to silence?
2. Are these trajectories associated with audience similarity and negative references in both directions during the baseline period?
3. Are shared negative targets associated with different trajectories?

A hyperlink records a reference from a source community to a target community. A negative label indicates negativity toward the linked post; a nonnegative label includes both neutral and positive references. These observations alone do not establish attacks, friendship, or reconciliation.

## 1. Setup

Paths are relative to the repository root. Raw source files are stored in `data/raw/`.

In [2]:
from pathlib import Path

# Open this notebook from the repository root.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    raise RuntimeError("Open results.ipynb with readait as the working directory.")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"


## 2. Hyperlink data: a first look

The Reddit Hyperlink Network contains separate files for links appearing in post bodies and titles. We inspect the first 1,000 rows of each file, preserving the original values before any cleaning.

Each row records a hyperlink from a post in one subreddit (the **source**) to a post in another subreddit (the **target**).

| Column | Meaning |
|---|---|
| `SOURCE_SUBREDDIT` | Community containing the linking post |
| `TARGET_SUBREDDIT` | Community containing the linked post |
| `POST_ID` | Identifier of the source post |
| `TIMESTAMP` | Publication time of the source post |
| `LINK_SENTIMENT` | −1 for negativity toward the linked post; +1 for neutral or positive references |
| `PROPERTIES` | 86 comma-separated numerical features of the source post |

The release does not contain the target-post identifier or the original text of either post.

In [6]:
import pandas as pd
from IPython.display import display

from src.data.loaders import load_link_sample, load_embedding_sample

body_sample = load_link_sample(RAW_DATA_DIR / "soc-redditHyperlinks-body.tsv")
title_sample = load_link_sample(RAW_DATA_DIR / "soc-redditHyperlinks-title.tsv")

print("Body hyperlinks")
display(body_sample.head())
print("Title hyperlinks")
display(title_sample.head())

Body hyperlinks


,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,leagueoflegends,teamredditteams,1u4nrps,2013-12-31 16:39:58,1,"345.0,298.0,0.75652173913,0.0173913043478,0.08..."
1,theredlion,soccer,1u4qkd,2013-12-31 18:18:37,-1,"101.0,98.0,0.742574257426,0.019801980198,0.049..."
2,inlandempire,bikela,1u4qlzs,2014-01-01 14:54:35,1,"85.0,85.0,0.752941176471,0.0235294117647,0.082..."
3,nfl,cfb,1u4sjvs,2013-12-31 17:37:55,1,"1124.0,949.0,0.772241992883,0.0017793594306,0...."
4,playmygame,gamedev,1u4w5ss,2014-01-01 02:51:13,1,"715.0,622.0,0.777622377622,0.00699300699301,0...."


Title hyperlinks


,SOURCE_SUBREDDIT,TARGET_SUBREDDIT,POST_ID,TIMESTAMP,LINK_SENTIMENT,PROPERTIES
0,rddtgaming,rddtrust,1u4pzzs,2013-12-31 16:39:18,1,"25.0,23.0,0.76,0.0,0.44,0.12,0.12,4.0,4.0,0.0,..."
1,xboxone,battlefield_4,1u4tmfs,2013-12-31 17:59:11,1,"100.0,88.0,0.78,0.02,0.08,0.13,0.07,16.0,16.0,..."
2,ps4,battlefield_4,1u4tmos,2013-12-31 17:59:40,1,"100.0,88.0,0.78,0.02,0.08,0.13,0.07,16.0,16.0,..."
3,fitnesscirclejerk,leangains,1u50xfs,2013-12-31 19:01:56,1,"49.0,43.0,0.775510204082,0.0,0.265306122449,0...."
4,fitnesscirclejerk,lifeprotips,1u51nps,2013-12-31 21:02:28,1,"14.0,14.0,0.785714285714,0.0,0.428571428571,0...."


In [9]:
sample_overview = pd.DataFrame({
    "body_dtype": body_sample.dtypes.astype(str),
    "title_dtype": title_sample.dtypes.astype(str),
})
display(sample_overview)
print(f"Preview rows: {len(body_sample):,} body; {len(title_sample):,} title")
print("Same columns in both files:", body_sample.columns.equals(title_sample.columns))

,body_dtype,title_dtype
SOURCE_SUBREDDIT,str,str
TARGET_SUBREDDIT,str,str
POST_ID,str,str
TIMESTAMP,str,str
LINK_SENTIMENT,str,str
PROPERTIES,str,str


Preview rows: 1,000 body; 1,000 title
Same columns in both files: True


### Subreddit embeddings

The embedding CSV has no header. Each row contains a subreddit name followed by 300 numerical coordinates derived from user participation across communities. We name these coordinates `embedding_1` through `embedding_300`; individual coordinates have no assigned topic or sentiment meaning.

These vectors describe communities, whereas the hyperlink `PROPERTIES` describe source posts. Similar vectors reflect similar participation patterns, not necessarily agreement or friendship.


In [11]:
embedding_sample = load_embedding_sample(
    RAW_DATA_DIR / "web-redditEmbeddings-subreddits.csv"
)

# Show a few coordinates to keep the preview readable.
display(embedding_sample.iloc[:5, :6])
print(f"Preview rows: {len(embedding_sample):,}")
print(f"Embedding coordinates: {embedding_sample.shape[1] - 1}")
print("Subreddit dtype:", embedding_sample["subreddit"].dtype)
print("Coordinate dtypes:", embedding_sample.iloc[:, 1:].dtypes.unique())

,subreddit,embedding_1,embedding_2,embedding_3,embedding_4,embedding_5
0,spiders,0.158972,0.285813,0.226329,-0.183338,-0.163159
1,askreddit,-0.499114,0.323983,-0.424809,-0.222705,-0.327201
2,globaloffensivetrade,-0.023145,-1.199374,1.661484,-1.025296,1.424670
3,fireteams,2.492506,-2.529917,-0.448484,-3.543441,-0.586122
4,funny,-0.819370,-0.865261,0.301753,0.018787,0.201029


Preview rows: 1,000
Embedding coordinates: 300
Subreddit dtype: str
Coordinate dtypes: [dtype('float64')]
